In [1]:
from __future__ import annotations

from pathlib import Path
import os
import json
import random

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

from st_pipeline.constants import KEYS
from st_pipeline.superres.feature_cache import (
    ensure_feature_cache,
    get_common_genes,
    load_cache_slide,
    normalize_slide,
)
from st_pipeline.data.gene_vocab import load_gene_vocab, map_genes_to_vocab
from st_pipeline.data.collate import mil_collate
from st_pipeline.model.morpho_cellfm_mil import MorphoCellfmMIL
from st_pipeline.model.nb_loss import nb_negative_log_likelihood

# ----------------------------
# 0) Config
# ----------------------------
ROOT = Path(os.environ.get("MORPHO_VC_ROOT", r"e:\Morpho-VC")).resolve()
CKPT_DIR = ROOT / "checkpoints" / "st_superres" / "xenium_cellfm_2"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

gene_vocab_path = ROOT / "assets" / "cellfm" / "expand_gene_info.csv"
cellfm_checkpoint = ROOT / "checkpoints" / "CellFM" / "CellFM_80M_weight.pt"

# Train/val/test split
train_ids = ["XEN2"]
val_ids = ["XEN2"]  # keep non-empty to avoid dataset issues
test_id = "XEN1"

# Training hyper-params
seed = 42
batch_size = 40
num_workers = 0
epochs = 100
lr = 3e-4
gene_chunk_size = 512
use_amp = True
heartbeat_batches = 5  # print every N batches

# Resume/skip behavior (notebook-friendly)
# - If best_model.pt exists:
#   - force_train=True: ignore checkpoint, train from scratch
#   - resume_if_exists=True: load checkpoint and continue
#   - otherwise: skip training
resume_if_exists = True
force_train = False

# Cache params
force_recompute = False
max_dim = 16000

# ----------------------------
# 1) Repro + device
# ----------------------------
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ----------------------------
# 2) Ensure cache + gene list
# ----------------------------
ensure_feature_cache(
    slide_ids=train_ids + val_ids,
    root=ROOT,
    force=force_recompute,
    device=device,
    max_dim=max_dim,
    use_tissue_seg=True,
)

if (CKPT_DIR / "common_genes.json").exists():
    common_genes = json.loads((CKPT_DIR / "common_genes.json").read_text())
else:
    common_genes = get_common_genes(train_ids + val_ids, root=ROOT)

vocab = load_gene_vocab(gene_vocab_path)
kept_genes, gene_ids = map_genes_to_vocab(common_genes, vocab)
(CKPT_DIR / "common_genes.json").write_text(json.dumps(kept_genes, indent=2), encoding="utf-8")


# ----------------------------
# 3) Dataset
# ----------------------------
def get_disk_mask(radius):
    radius_ceil = int(np.ceil(radius))
    locs = np.meshgrid(
        np.arange(-radius_ceil, radius_ceil + 1),
        np.arange(-radius_ceil, radius_ceil + 1),
        indexing="ij",
    )
    locs = np.stack(locs, axis=-1)
    distsq = (locs ** 2).sum(-1)
    return distsq <= radius ** 2


GRID_FACTOR = 16  # embedding grid is 1/16 of HE pixels


class SuperresSpotDataset(Dataset):
    def __init__(self, embs, cnts, locs_px, radius_px, gene_ids, grid_factor=GRID_FACTOR):
        super().__init__()
        self.embs = embs
        self.cnts = cnts
        self.gene_ids = gene_ids

        locs_px = np.asarray(locs_px, dtype=np.float32)
        self.locs_px = locs_px.copy()
        locs_grid = np.round(locs_px / float(grid_factor)).astype(np.int64)

        # Convert radius (pixel) -> grid space
        self.radius = float(radius_px) / float(grid_factor)
        self.mask = get_disk_mask(self.radius)
        self.mask_size = int(self.mask.sum())

        H, W = embs.shape[:2]
        r0, r1 = self._patch_offsets()
        valid = (
                (locs_grid[:, 0] + r0[0] >= 0) &
                (locs_grid[:, 0] + r0[1] <= H) &
                (locs_grid[:, 1] + r1[0] >= 0) &
                (locs_grid[:, 1] + r1[1] <= W)
        )
        finite_center = np.isfinite(embs[locs_grid[:, 0], locs_grid[:, 1]]).all(-1)
        valid &= finite_center

        self.keep_idx = np.where(valid)[0]
        self.locs = locs_grid[valid]
        self.locs_px = self.locs_px[valid]
        self.cnts = cnts[valid]

        # Size factors based on total counts
        sums = self.cnts.sum(axis=1)
        self.size_factors = (sums / (sums.mean() + 1e-8)).astype(np.float32)

    def _patch_offsets(self):
        shape = np.array(self.mask.shape)
        center = shape // 2
        r = np.stack([-center, shape - center], -1)
        return r[0], r[1]

    def __len__(self):
        return len(self.locs)

    def __getitem__(self, idx):
        i, j = self.locs[idx]
        r0, r1 = self._patch_offsets()
        patch = self.embs[
            i + r0[0]: i + r0[1],
            j + r1[0]: j + r1[1]
        ]
        if self.mask.all():
            x = patch
        else:
            x = patch[self.mask]
        x = torch.from_numpy(x.astype(np.float32))
        y = torch.from_numpy(self.cnts[idx].astype(np.float32))
        size_factor = float(self.size_factors[idx])
        return {
            KEYS.X: x,
            KEYS.Y_BAG: y,
            KEYS.SIZE_FACTOR: size_factor,
            KEYS.GENE_IDS: self.gene_ids,
        }


# Build datasets
datasets = {}
slide_stats = {}
for sid in train_ids + val_ids:
    print(f"Loading slide {sid} ...")
    embs, cnts, locs, radius, _, _ = load_cache_slide(sid, root=ROOT, genes=kept_genes)
    embs_norm, _, stats = normalize_slide(embs, cnts)
    ds = SuperresSpotDataset(embs_norm, cnts, locs, radius, gene_ids)
    datasets[sid] = ds
    slide_stats[sid] = {
        "radius_px": float(radius),
        "mask_size": int(ds.mask_size),
        "embs_mean": stats["embs_mean"],
        "embs_std": stats["embs_std"],
    }
    print(f"  embs: {embs.shape} | cnts: {cnts.shape} | locs: {locs.shape}")
    print(f"  radius (px): {radius:.2f} | mask_size: {ds.mask_size}")
    print(f"  dataset size: {len(ds)}")

for sid, st in slide_stats.items():
    out = CKPT_DIR / f"stats_{sid}.npz"
    np.savez(out, **st)
    print(f"Saved stats: {out}")

# ----------------------------
# 4) Train
# ----------------------------
train_ds = ConcatDataset([datasets[sid] for sid in train_ids])
val_ds = ConcatDataset([datasets[sid] for sid in val_ids])

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    collate_fn=mil_collate,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(num_workers > 0),
)
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    collate_fn=mil_collate,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(num_workers > 0),
)

sample_batch = next(iter(train_loader))
input_dim = sample_batch[KEYS.X].shape[-1]

model = MorphoCellfmMIL(
    input_dim=input_dim,
    n_genes=len(kept_genes),
    cellfm_dim=1536,
    cellfm_layers=2,
    cellfm_heads=48,
    cellfm_checkpoint=str(cellfm_checkpoint),
    freeze_cellfm=True,
    use_mock=False,
    use_retention=True,
    vocab_size=len(vocab),
    dropout=0.1,
    aggregation="mean",
    dispersion="gene",
).to(device)

# Freeze CellFM backbone; train gene-side + adapter
for p in model.cellfm.model.parameters():
    p.requires_grad = False
model.cellfm.model.gene_emb.requires_grad = True
for p in model.cellfm.model.cellwise_dec.map.parameters():
    p.requires_grad = True
model.gene_dispersion.requires_grad = True

ckpt_path = CKPT_DIR / "best_model.pt"
skip_training = False
if ckpt_path.exists():
    if force_train:
        print(f"Checkpoint exists ({ckpt_path}) but force_train=True -> train from scratch.")
    elif resume_if_exists:
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        print(f"Checkpoint exists -> resume training from {ckpt_path}")
    else:
        print(f"Checkpoint exists ({ckpt_path}) and resume_if_exists=False -> skip training.")
        skip_training = True
else:
    print("No checkpoint found -> training from scratch.")

optimizer = torch.optim.AdamW(
    [
        {"params": [model.cellfm.model.gene_emb], "lr": lr * 3.0},
        {"params": model.cellfm.model.cellwise_dec.map.parameters(), "lr": lr * 3.0},
        {"params": model.adapter.parameters(), "lr": lr},
        {"params": [model.gene_dispersion], "lr": lr},
    ],
    weight_decay=1e-2,
)

use_amp = bool(use_amp and device.type == "cuda")
scaler = torch.amp.GradScaler(enabled=use_amp)

# Gene chunking (safe for memory)
gene_count = len(kept_genes)
gene_chunks = [list(range(i, min(i + gene_chunk_size, gene_count)))
               for i in range(0, gene_count, gene_chunk_size)]

best_val = 1e9
no_improve = 0
early_stop_patience = 5
early_stop_min_delta = 1e-4

if not skip_training:
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        steps = 0

        train_iter = tqdm(train_loader, total=len(train_loader), desc=f"Epoch {epoch} train",
                          ncols=100) if tqdm else train_loader
        for batch_idx, batch in enumerate(train_iter):
            for k in batch:
                batch[k] = batch[k].to(device)

            chunk_order = torch.randperm(len(gene_chunks)).tolist()
            for ci in chunk_order:
                idx = torch.as_tensor(gene_chunks[ci], device=device)
                batch_chunk = {
                    KEYS.X: batch[KEYS.X],
                    KEYS.PTR_BAG_INSTANCE: batch[KEYS.PTR_BAG_INSTANCE],
                    KEYS.SIZE_FACTOR: batch[KEYS.SIZE_FACTOR],
                    KEYS.GENE_IDS: batch[KEYS.GENE_IDS][idx],
                    KEYS.Y_BAG: batch[KEYS.Y_BAG][:, idx],
                }

                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                    mu_bag, _ = model(batch_chunk)
                    theta = torch.exp(model.gene_dispersion[idx])
                    loss = nb_negative_log_likelihood(batch_chunk[KEYS.Y_BAG], mu_bag, theta)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                total += float(loss.item())
                steps += 1

            if tqdm:
                train_iter.set_postfix(loss=f"{loss.item():.4f}")
            elif batch_idx % heartbeat_batches == 0:
                print(f"Epoch {epoch} batch {batch_idx}/{len(train_loader)}", flush=True)

        train_loss = total / max(steps, 1)

        # Validation
        model.eval()
        total = 0.0
        steps = 0
        with torch.no_grad():
            val_iter = tqdm(val_loader, total=len(val_loader), desc=f"Epoch {epoch} val", ncols=100,
                            leave=False) if tqdm else val_loader
            for batch in val_iter:
                for k in batch:
                    batch[k] = batch[k].to(device)
                for ci in range(len(gene_chunks)):
                    idx = torch.as_tensor(gene_chunks[ci], device=device)
                    batch_chunk = {
                        KEYS.X: batch[KEYS.X],
                        KEYS.PTR_BAG_INSTANCE: batch[KEYS.PTR_BAG_INSTANCE],
                        KEYS.SIZE_FACTOR: batch[KEYS.SIZE_FACTOR],
                        KEYS.GENE_IDS: batch[KEYS.GENE_IDS][idx],
                        KEYS.Y_BAG: batch[KEYS.Y_BAG][:, idx],
                    }
                    with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                        mu_bag, _ = model(batch_chunk)
                        theta = torch.exp(model.gene_dispersion[idx])
                        loss = nb_negative_log_likelihood(batch_chunk[KEYS.Y_BAG], mu_bag, theta)
                    total += float(loss.item())
                    steps += 1
                if tqdm:
                    val_iter.set_postfix(loss=f"{loss.item():.4f}")

        val_loss = total / max(steps, 1)
        print(f"Epoch {epoch}: train={train_loss:.4f} val={val_loss:.4f}")

        if val_loss < best_val - early_stop_min_delta:
            best_val = val_loss
            no_improve = 0
            torch.save(model.state_dict(), CKPT_DIR / "best_model.pt")
            (CKPT_DIR / "split.json").write_text(json.dumps({
                "train": train_ids,
                "val": val_ids,
                "test": [test_id]
            }, indent=2), encoding="utf-8")
            print(f"Saved best model -> {CKPT_DIR / 'best_model.pt'}")
        else:
            no_improve += 1
            if no_improve >= early_stop_patience:
                print("Early stop.")
                break

print("Training done.")
